In [5]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'

# 加载模型和 tokenizer（从本地加载）
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# 将模型转移到GPU（如果可用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)

# 读取 CSV 文件中的数据
data_dir = r'/home/wangshuo/resource/datasets/workload4'
csv_file = data_dir + '/posts.csv'
print(csv_file)
df = pd.read_csv(csv_file)

# 获取所有的 "body" 字段内容
all_posts = df['body'].tolist()
print(f"数据加载完毕，共有 {len(all_posts)} 个帖子")

# 定义类别列表
categories = [
    "Politics", "Entertainment", "Technology",  "Economy", "Culture", "Travel"
]

# 定义推理函数
def check_nli_batch(topic, batch_posts):
    """
    对每一批次的 post 进行 NLI 推理，判断其是否与给定的 topic 匹配
    """
    # 创建每个输入样本的 "topic + post" 对
    batch_inputs = [f"{post} [SEP] {topic}" for post in batch_posts]

    # 编码批量输入
    encoding = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 模型推理
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 选择 "entailment" 和 "contradiction" 的 logits
        entail_contradiction_logits = logits[:, [0, 2]]

        # 计算 softmax 概率
        probs = entail_contradiction_logits.softmax(dim=1)

        # 获取 "entailment" 类别的概率（标签为真时的概率）
        prob_label_is_true = probs[:, 1].cpu().numpy()
        
        return prob_label_is_true

# 假设每批次处理 32 个文本
batch_size = 32
results = []

# 使用 tqdm 显示进度条
for i in tqdm(range(0, len(all_posts), batch_size), desc="Processing Posts"):
    batch_posts = all_posts[i:i + batch_size]
    
    # 对每个类别进行推理
    category_probs = {category: [] for category in categories}
    
    # 遍历所有类别
    for category in categories:
        probabilities = check_nli_batch(category, batch_posts)
        category_probs[category] = probabilities
    
    # 对每个帖子，选择概率最大对应的类别
    for j in range(len(batch_posts)):
        max_category = max(category_probs, key=lambda x: category_probs[x][j])
        results.append(max_category)

# 将结果添加到 DataFrame
df['bart_category_predicted'] = results

# 保存包含新列的 CSV 文件
output_csv_file = data_dir + '/static/posts_with_predicted_categories.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


cuda
/home/wangshuo/resource/datasets/workload4/posts.csv
数据加载完毕，共有 8906 个帖子


Processing Posts: 100%|██████████| 279/279 [10:10<00:00,  2.19s/it]


OSError: Cannot save file into a non-existent directory: '/home/wangshuo/resource/datasets/workload4/static'

In [6]:
output_csv_file = data_dir + '/static/posts_with_predicted_categories.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")

推理结果已保存到 /home/wangshuo/resource/datasets/workload4/static/posts_with_predicted_categories.csv


统计各个类别的数量

In [8]:
import pandas as pd

# 读取 CSV 文件
csv_file = '/home/wangshuo/resource/datasets/workload4/static/posts_with_predicted_categories.csv'
df = pd.read_csv(csv_file)

# 检查数据的前几行，以确认 'predicted_category' 列存在
print(df.head())

# 统计各个类别的数量
category_counts = df['bart_category_predicted'].value_counts()

# 输出各个类别的数量
print(category_counts)

# 如果需要将结果保存到文件中，可以使用以下命令：
output_file = '/home/wangshuo/resource/datasets/workload_20w/category_counts.csv'
category_counts.to_csv(output_file, header=True)

print(f"类别统计已保存到 {output_file}")


  :LABEL                             id:ID  comments  \
0   Post  33716ea8b17a4cb48b0390851744f0b1      6000   
1   Post  8abcda8edbef446a997479eee7e74a25      3300   
2   Post  cf05771038b545c1a6353a80c6a3fa59      3600   
3   Post  1b71b627f5ac4566802481533dbe6556      2600   
4   Post  7d4dea13d3cd46faa26bb0bab5fe886a      2300   

                                                body     createdAt  \
0  Schiff Refuses To Disclose Why He Withheld Det...  2.020121e+13   
1  Trump asks 'Where's Durham?' during first inte...  2.020113e+13   
2  The documentation of my claims about Justices ...  2.020122e+13   
3  I'm going to do everything I can to try to pre...  2.020112e+13   
4  Last year at Thanksgiving, I told my children ...  2.020113e+13   

                            creator  score  sensitive  upvotes  \
0  cd346addd6534b43adc2a06a572ce0f5    0.0      False    19000   
1  cd346addd6534b43adc2a06a572ce0f5    0.0      False    13000   
2  2ea66900ed69a5005fe51fc8b07a1711    0.0  

comments_with_nli_results.csv先按照属性bart_entailment_probability排序，再按照pcNum字段排序

In [13]:
import pandas as pd

# 读取 CSV 文件
csv_file = '/home/wangshuo/resource/datasets/workload4/static/posts_with_predicted_categories.csv'
df = pd.read_csv(csv_file)

# 按照 'bart_entailment_probability' 降序排序，再按照 'pcNum' 降序排序
df_sorted = df.sort_values(by=['bart_category_predicted', 'pcNum'], ascending=[False, False])

# 保存排序后的结果到新的 CSV 文件
output_file = '/home/wangshuo/resource/datasets/workload4/static/posts_with_predicted_categories.csv'
df_sorted.to_csv(output_file, index=False)

print(f"排序后的数据已保存到 {output_file}")


排序后的数据已保存到 /home/wangshuo/resource/datasets/workload4/static/posts_with_predicted_categories.csv


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
import numpy as np

# 定义类别及其描述
CATEGORIES = {
    "Politics": "This text is about politics, political news, political views, or political figures",
    "Entertainment": "This text is about movies, TV shows, celebrities, music, or entertainment industry",
    "Sports": "This text is about sports events, athletes, sports teams, or sports activities",
    "Technology": "This text is about technology products, innovation, software, hardware, or tech industry",
    "Health": "This text is about health, diseases, fitness, or medical topics",
    "Economy": "This text is about economy, finance, stock market, or business",
    "Culture": "This text is about culture, art, history, or cultural topics",
    "Travel": "This text is about travel, tourist attractions, or travel experiences"
}

def load_model_and_tokenizer(model_path):
    """加载模型和分词器"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    return model, tokenizer, device

def prepare_hypothesis(text, category_description):
    """准备假设文本"""
    return f"{text} [SEP] {category_description}"

def batch_classify(texts, category_descriptions, model, tokenizer, device, batch_size=32):
    """批量处理文本分类"""
    all_scores = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        batch_scores = []
        
        for description in category_descriptions:
            # 为每个类别准备假设
            hypotheses = [prepare_hypothesis(text, description) for text in batch_texts]
            
            # 编码
            inputs = tokenizer(
                hypotheses,
                padding=True,
                truncation=True,
                max_length=1024,
                return_tensors="pt"
            )
            
            # 移到GPU
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # 推理
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
                # 获取 entailment (标签为真) 的概率
                probs = torch.softmax(logits, dim=1)[:, 1]
                batch_scores.append(probs.cpu().numpy())
        
        # 将每个批次的分数转换为数组
        batch_scores = np.array(batch_scores).T
        all_scores.extend(batch_scores)
    
    return np.array(all_scores)

def main():
    # 配置路径
    model_path = "/path/to/bart-large-mnli"  # 替换为你的模型路径
    csv_path = "/path/to/your/data.csv"      # 替换为你的CSV文件路径
    output_path = "/path/to/output.csv"      # 替换为输出文件路径
    
    # 加载模型
    print("Loading model...")
    model, tokenizer, device = load_model_and_tokenizer(model_path)
    
    # 加载数据
    print("Loading data...")
    df = pd.read_csv(csv_path)
    texts = df['body'].tolist()
    
    # 准备类别描述
    category_descriptions = list(CATEGORIES.values())
    category_names = list(CATEGORIES.keys())
    
    # 批量处理分类
    print("Processing classification...")
    scores = batch_classify(texts, category_descriptions, model, tokenizer, device)
    
    # 获取最高分的类别
    predicted_categories = [category_names[score.argmax()] for score in scores]
    category_scores = [score.max() for score in scores]
    
    # 添加结果到DataFrame
    df['predicted_category'] = predicted_categories
    df['category_confidence'] = category_scores
    
    # 保存结果
    df.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")
    
    # 打印分类统计
    print("\nCategory Distribution:")
    print(df['predicted_category'].value_counts())

if __name__ == "__main__":
    main()